In [108]:
import os

import numpy as np
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

import torch
import torchvision
from torchvision.transforms import transforms

from PIL import Image

from preprocessing_module import find_class_names_filenames, stratified_split_data_paths, NatureCityScenesDataset

In [ ]:
import os
import sys
import random
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

from sklearn.model_selection import train_test_split


# ---- Project paths ----
PROJECT_ROOT = Path("quadrant_dots_project").resolve()
DATA_ROOT = PROJECT_ROOT / ".." / ".." / "Datasets" / "quadrant_dots_rgb"

# Make project importable (so we can import our dataset + generator modules)
sys.path.append(str(PROJECT_ROOT))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)


Device: cpu


In [110]:
from sklearn.metrics import average_precision_score

In [115]:
DATASET_PATH = r"./mandatory1_data"

class_names, class_filenames = find_class_names_filenames(DATASET_PATH)
x_train_paths, x_val_paths, x_test_paths, y_train, y_val, y_test = stratified_split_data_paths(DATASET_PATH, class_names, class_filenames)

transform = transforms.Compose([
    transforms.Resize((224, 224)), # AlexNet standard size: 224x224
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_set = NatureCityScenesDataset(x_train_paths, y_train, transform=transform)
val_set = NatureCityScenesDataset(x_val_paths, y_val, transform=transform)
test_set = NatureCityScenesDataset(x_test_paths, y_test, transform=transform)

BATCH_SIZE = 32
train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)




subset_indices = list(range(100))
train_subset = torch.utils.data.Subset(train_set, subset_indices)
val_subset = torch.utils.data.Subset(val_set, subset_indices)
test_subset = torch.utils.data.Subset(test_set, subset_indices)

train_loader = torch.utils.data.DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = torch.utils.data.DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)

In [112]:
from dataclasses import dataclass
from typing import Callable, Dict, Optional, Tuple

@dataclass
class EpochStats:
    loss: float
    acc: float
    mAP: float


class Trainer:
    def __init__(
        self,
        model: nn.Module,
        criterion: nn.Module,
        optimizer: torch.optim.Optimizer,
        device: torch.device,
        augment_fn: Optional[Callable[[torch.Tensor], torch.Tensor]] = None,
    ):
        self.model = model
        self.criterion = criterion
        self.optimizer = optimizer
        self.device = device
        self.augment_fn = augment_fn

        self.history: Dict[str, list] = {
            "train_loss": [],
            "train_acc": [],
            "val_loss": [],
            "val_acc": [],
        }

    @staticmethod
    def _accuracy(logits: torch.Tensor, targets: torch.Tensor) -> float:
        preds = logits.argmax(dim=1)
        return (preds == targets).float().mean().item()

    def train_one_epoch(self, loader: DataLoader) -> EpochStats:
        self.model.train()

        total_loss = 0.0
        total_acc = 0.0
        n_batches = 0

        for images, targets in loader:
            images = images.to(self.device, non_blocking=True)
            targets = targets.to(self.device, non_blocking=True)

            # Apply augmentation policy only during training (optional)
            if self.augment_fn is not None:
                images = self.augment_fn(images)

            logits = self.model(images)
            loss = self.criterion(logits, targets)

            self.optimizer.zero_grad(set_to_none=True)
            loss.backward()
            self.optimizer.step()

            total_loss += loss.item()
            total_acc += self._accuracy(logits, targets)
            n_batches += 1

        return EpochStats(loss=total_loss / n_batches, acc=total_acc / n_batches, mAP=0.0)

    @torch.no_grad()
    def evaluate(self, loader: DataLoader) -> EpochStats:
        self.model.eval()

        total_loss = 0.0
        total_acc = 0.0
        n_batches = 0

        store_logits = []
        store_targets = []

        for images, targets in loader:
            images = images.to(self.device, non_blocking=True)
            targets = targets.to(self.device, non_blocking=True)

            logits = self.model(images)
            loss = self.criterion(logits, targets)

            total_loss += loss.item()
            total_acc += self._accuracy(logits, targets)
            n_batches += 1

            store_logits.append(logits)
            store_targets.append(targets)

        # logits have shape [batch size, number of classes], 
        # first row is the raw output from the model for the first individual sample of the batch.
        # Turn list of tensors (one for each batch) into one tensor-typed list
        store_logits = torch.concatenate(store_logits)
        store_targets = torch.concatenate(store_targets)

        # Turn raw model output into probabilities with softmax, dim=1 so we get sofmax per indv. sample over all classes
        prob_scores = torch.softmax(store_logits, dim=1)
        # Turn target values (class number) into one-hot encoded tensor 
        one_hot_targets = torch.nn.functional.one_hot(store_targets)
        mean_avg_precision = average_precision_score(y_score=prob_scores, y_true=one_hot_targets)

        return EpochStats(loss=total_loss/n_batches, acc=total_acc/n_batches, mAP=mean_avg_precision)

    def fit(self, train_loader: DataLoader, val_loader: DataLoader, epochs: int = 10):
        for epoch in range(1, epochs + 1):
            train_stats = self.train_one_epoch(train_loader)
            val_stats = self.evaluate(val_loader)

            self.history["train_loss"].append(train_stats.loss)
            self.history["train_acc"].append(train_stats.acc)
            self.history["val_loss"].append(val_stats.loss)
            self.history["val_acc"].append(val_stats.acc)

            print(
                f"Epoch {epoch:02d} | "
                f"train loss: {train_stats.loss:.4f}, acc: {train_stats.acc:.3f} | "
                f"val loss: {val_stats.loss:.4f}, acc: {val_stats.acc:.3f}, average precision: {val_stats.mAP}"
            )


In [113]:
def batch_noise_augment(images: torch.Tensor, std: float = 0.03) -> torch.Tensor:
    noise = torch.randn_like(images) * std
    return torch.clamp(images + noise, 0.0, 1.0)

from ResNet import ResNet

model_ResNet18 = ResNet(img_channels=3, num_layers=18, num_classes=6).to(device)

num_epochs = 4

In [114]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_ResNet18.parameters(), lr=1e-3)

trainer_ResNet18_1 = Trainer(
    model=model_ResNet18,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    augment_fn=lambda x: batch_noise_augment(x, std=0.02),
)

trainer_ResNet18_1.fit(train_loader, val_loader, epochs=num_epochs)

Epoch 01 | train loss: 2.5156, acc: 0.156 | val loss: 1.7866, acc: 0.133, average precision: 0.2889530982193939
Epoch 02 | train loss: 1.8631, acc: 0.320 | val loss: 1.8265, acc: 0.266, average precision: 0.2834134300692754
Epoch 03 | train loss: 1.5133, acc: 0.328 | val loss: 2.7435, acc: 0.188, average precision: 0.30343134657385823
Epoch 04 | train loss: 1.5121, acc: 0.391 | val loss: 3.6976, acc: 0.195, average precision: 0.24655444701390383


In [88]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_ResNet18.parameters(), lr=1e-2)

trainer_ResNet18_2 = Trainer(
    model=model_ResNet18,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    augment_fn=lambda x: batch_noise_augment(x, std=0.02),
)

trainer_ResNet18_2.fit(train_loader, val_loader, epochs=num_epochs)

Epoch 01 | train loss: 4.1602, acc: 0.094 | val loss: 39660410.0000, acc: 0.211, average precision: 0.16666666666666666


KeyboardInterrupt: 

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.RMSprop(model_ResNet18.parameters(), lr=0.03)

trainer_ResNet18_2 = Trainer(
    model=model_ResNet18,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    augment_fn=lambda x: batch_noise_augment(x, std=0.02),
)

trainer_ResNet18_2.fit(train_loader, val_loader, epochs=num_epochs)

Epoch 01 | train loss: 9.9087, acc: 0.125 | val loss: 1213929.4688, acc: 0.211, average precision: 0.16666666666666666
Epoch 02 | train loss: 4.9907, acc: 0.242 | val loss: 4176.9534, acc: 0.141, average precision: 0.18796296296296297
Epoch 03 | train loss: 2.6442, acc: 0.141 | val loss: 161.8815, acc: 0.188, average precision: 0.20642956025709583
Epoch 04 | train loss: 2.0320, acc: 0.180 | val loss: 34.0463, acc: 0.094, average precision: 0.21065217837340788


In [8]:
trainer.history

{'train_loss': [2.517153799533844,
  1.9044808745384216,
  1.4978394508361816,
  1.7311114072799683],
 'train_acc': [0.1796875, 0.296875, 0.4140625, 0.4296875],
 'val_loss': [1.6763566136360168,
  1.8834748268127441,
  2.257263943552971,
  1.7065287232398987],
 'val_acc': [0.25, 0.28125, 0.4765625, 0.3828125]}

In [78]:
trainer.evaluate(test_loader)

EpochStats(loss=24.562232971191406, acc=0.1796875, ap=0.26409306931902526)

In [33]:
stored_imgs_list = []
stored_labels_list = []
for i, j in train_loader:
    stored_imgs_list.append(i)
    stored_labels_list.append(j)
    

In [34]:
stored_labels_list

[tensor([3, 3, 2, 2, 1, 3, 0, 2, 3, 4, 2, 4, 5, 3, 2, 1, 1, 3, 3, 2, 1, 3, 2, 1,
         3, 3, 1, 5, 5, 5, 0, 2]),
 tensor([2, 5, 1, 0, 4, 2, 5, 0, 1, 4, 1, 4, 0, 2, 0, 0, 1, 5, 4, 3, 2, 1, 1, 0,
         4, 3, 3, 4, 5, 5, 2, 1]),
 tensor([2, 1, 5, 1, 0, 2, 0, 2, 5, 5, 1, 5, 5, 4, 0, 2, 3, 4, 1, 0, 2, 1, 2, 0,
         0, 2, 5, 0, 0, 0, 5, 0]),
 tensor([4, 1, 4, 2])]

In [79]:
torch.concatenate(stored_labels_list, dim=0)

tensor([3, 3, 2, 2, 1, 3, 0, 2, 3, 4, 2, 4, 5, 3, 2, 1, 1, 3, 3, 2, 1, 3, 2, 1,
        3, 3, 1, 5, 5, 5, 0, 2, 2, 5, 1, 0, 4, 2, 5, 0, 1, 4, 1, 4, 0, 2, 0, 0,
        1, 5, 4, 3, 2, 1, 1, 0, 4, 3, 3, 4, 5, 5, 2, 1, 2, 1, 5, 1, 0, 2, 0, 2,
        5, 5, 1, 5, 5, 4, 0, 2, 3, 4, 1, 0, 2, 1, 2, 0, 0, 2, 5, 0, 0, 0, 5, 0,
        4, 1, 4, 2])

In [74]:
torch.nn.functional.one_hot(torch.concatenate(stored_labels_list, dim=0))

tensor([[0, 0, 0, 1, 0, 0],
        [0, 0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0],
        [1, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 1, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 1],
        [0, 0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 1, 0, 0, 0, 0],
        [0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0],
        [0, 0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0],
        [0, 0, 0, 1, 0, 0],
        [0, 1, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 1],
        [0, 0, 0, 0, 0, 1],
        [0, 0, 0, 0, 0, 1],
        [1, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 0, 1],
        [0, 1, 0, 0, 0, 0],
        [1, 0, 0, 0,

In [ ]:
average_precision_score(y_score=[0.25, 0.6, 0.15], y_true=[0, 0, 1])

0.3333333333333333